# Expand the dependence family

Dependence was the strongest leave-one-family-out helper (-0.008) in
`select_streaming_features.ipynb`, matching the batch oracle (AR(5) 0.639 alone).
Current dependence feats: `ac1_online`, `ac1_shift`, `ar_resid_logratio` (AR5).

Here we add streaming **higher-lag autocorr** (lags 2,3,5,10, online + shift vs
reference) and **AR-residual logratios at orders 2 and 10**, on top of the shipped
33-feat spectral set (submit#1-features_improved). Then: gain ranking, is the
expanded dependence family a net win (LOO), and which new lags/orders earn gain.

Same holdout protocol: 6000-series log-subsample train / 2000-series full-res holdout.

In [1]:
import math, time
from bisect import bisect_left, bisect_right
from collections import deque
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import roc_auc_score
PATH = "/Users/minqi/Documents/ADIA_Lab_Structural_Break_Challenge/"

## StreamState — 33 shipped feats + expanded dependence

New: `ac{2,3,5,10}_online`, `ac{2,3,5,10}_shift`, `ar{2,10}_resid_logratio` (10 feats). Streaming autocorr at lag L keeps a running sum of x_t·x_{t-L} via a maxlag ring buffer; extra AR orders each keep their own residual accumulator.

In [2]:

AC_LAGS_NEW = (2, 3, 5, 10)      # ac1 already in base
AR_ORDERS_NEW = (2, 10)          # ar5 already in base
MAXLAG = max(AC_LAGS_NEW)

def _autocorr(x, lag):
    if len(x) <= lag: return 0.0
    a, b = x[:-lag], x[lag:]
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a*a).sum() * (b*b).sum())
    return float((a*b).sum()/d) if d > 0 else 0.0

def _fit_ar(x, p=5):
    if len(x) <= p + 5: return None, 1.0
    rows = np.lib.stride_tricks.sliding_window_view(x, p+1)
    Y = rows[:, -1]
    Xd = np.column_stack([np.ones(len(rows)), rows[:, :-1][:, ::-1]])
    beta, *_ = np.linalg.lstsq(Xd, Y, rcond=None)
    resid = Y - Xd @ beta
    return beta, float(resid.var()) if len(resid) else 1.0

def _norm_psd(seg, win):
    x = (seg - seg.mean()) * win
    sp = np.fft.rfft(x)
    p = (sp.real**2 + sp.imag**2)[1:]
    s = p.sum()
    if s <= 0 or not np.isfinite(s): return None
    return p / s

def _spec_ec(p, freqs, logk):
    return -float((p*np.log(p+1e-12)).sum())*logk, float((freqs*p).sum())


class StreamState:
    VAR_MIN = 20; SPEC_W = 128
    _BASE = ("n_online","mean_shift_z","last_z","online_logvar","online_std",
        "online_skew","online_kurt","cumsum_range","cusum_max","tail2_frac",
        "tail3_frac","jump_freq","signrun_freq","ks_stat","rank_dev","ac1_online",
        "ac1_shift","ar_resid_logratio","ref_log_n","ref_skew","ref_kurt","ref_ac1")
    _SPEC = ("spec_entropy","spec_entropy_shift","spec_centroid_shift")

    def __init__(self, bins=16, roll_windows=(50,200), ewma_alphas=(0.05,0.2), ar_p=5):
        self.bins=bins; self.rw=tuple(roll_windows); self.alphas=tuple(ewma_alphas); self.p=ar_p
        fn = list(self._BASE)
        for a in self.alphas: fn += ["ewma%.2f_z"%a, "ewma%.2f_logvar"%a]
        for w in self.rw: fn += ["roll%d_meanshift"%w, "roll%d_logvar"%w]
        self._ew_off=len(self._BASE); self._roll_off=self._ew_off+2*len(self.alphas)
        self._spec_off=len(fn); fn += list(self._SPEC)
        # expanded dependence block
        self._dep_off=len(fn)
        self._dep_names=[]
        for L in AC_LAGS_NEW: self._dep_names += ["ac%d_online"%L, "ac%d_shift"%L]
        for p in AR_ORDERS_NEW: self._dep_names += ["ar%d_resid_logratio"%p]
        fn += self._dep_names
        self.feature_names=fn; self.ncol=len(fn)
        self._out=np.zeros(self.ncol); self._bin_targets=np.linspace(1.0/self.bins,1.0,self.bins)
        W=self.SPEC_W; self._spec_win=np.hanning(W)
        self._spec_freqs=np.arange(1,W//2+1,dtype=np.float64)/(W//2)
        self._spec_logk=1.0/math.log(len(self._spec_freqs))
        self.FAMILIES = {
            "moments":["n_online","mean_shift_z","last_z","online_logvar","online_std","online_skew","online_kurt"],
            "changepoint":["cumsum_range","cusum_max","ks_stat","rank_dev"],
            "tail_jump":["tail2_frac","tail3_frac","jump_freq","signrun_freq"],
            "dependence":["ac1_online","ac1_shift","ar_resid_logratio"]+self._dep_names,
            "ewma":["ewma%.2f_z"%a for a in self.alphas]+["ewma%.2f_logvar"%a for a in self.alphas],
            "rolling":["roll%d_meanshift"%w for w in self.rw]+["roll%d_logvar"%w for w in self.rw],
            "refctx":["ref_log_n","ref_skew","ref_kurt","ref_ac1"],
            "spectral":list(self._SPEC),
        }
        self.FAM_DEP_NEW = self._dep_names  # just the new ones

    def reset(self, reference):
        r=np.asarray(reference,dtype=np.float64)
        self.mu_h=float(r.mean()) if len(r) else 0.0
        self.sd_h=max(float(r.std(ddof=1)) if len(r)>1 else 1.0,1e-8)
        self.var_h=self.sd_h**2; self.ref_n=len(r); self._inv_refn=1.0/max(self.ref_n,1)
        z=(r-self.mu_h)/self.sd_h if len(r) else r
        self.ref_skew=float((z**3).mean()) if len(r) else 0.0
        self.ref_kurt=float((z**4).mean()-3.0) if len(r) else 0.0
        self.ref_ac1=_autocorr(r,1); self.ref_log_n=math.log(self.ref_n+1.0)
        self.ref_ac={L:_autocorr(r,L) for L in AC_LAGS_NEW}
        self.sorted_ref=np.sort(r).tolist() if len(r) else []
        edges=(np.quantile(r,np.linspace(0,1,self.bins+1)[1:-1]) if len(r) else np.zeros(self.bins-1))
        self.edges=edges.tolist()
        self.ar,self.ar_var_h=_fit_ar(r,self.p)
        self.arN={p:_fit_ar(r,p) for p in AR_ORDERS_NEW}
        self.ref_spec_ent=0.0; self.ref_spec_cen=0.0
        if len(r)>=self.SPEC_W:
            W=self.SPEC_W; acc=np.zeros(W//2); cnt=0
            for s in range(0,len(r)-W+1,W//2):
                p=_norm_psd(r[s:s+W],self._spec_win)
                if p is not None: acc+=p; cnt+=1
            if cnt:
                pr=acc/acc.sum(); self.ref_spec_ent,self.ref_spec_cen=_spec_ec(pr,self._spec_freqs,self._spec_logk)
        self.n=0; self.mean=self.M2=self.M3=self.M4=0.0
        self.sx=self.sxx=self.sxy=0.0
        self.cmax=self.cmin=self.cumsum=0.0; self.cusum_p=self.cusum_n=self.cusum_max=0.0
        self.tail2=self.tail3=0; self.binc=np.zeros(self.bins); self.rank_sum=0.0
        self.jumps=0; self.signruns=0; self.prev=None
        self.ar_resid_ss=0.0; self.ar_n=0; self.arbuf=deque(maxlen=self.p)
        self.roll={w:(deque(maxlen=w),[0.0,0.0]) for w in self.rw}
        self.ew={a:self.mu_h for a in self.alphas}; self.ewv={a:0.0 for a in self.alphas}
        self.specbuf=deque(maxlen=self.SPEC_W); self._last_spec_ent=0.0; self._last_spec_cen=0.0
        # expanded dependence state
        self.lagbuf=deque(maxlen=MAXLAG)      # recent raw values (most recent last)
        self.acP={L:0.0 for L in AC_LAGS_NEW}; self.acC={L:0 for L in AC_LAGS_NEW}
        self.arNbuf={p:deque(maxlen=p) for p in AR_ORDERS_NEW}
        self.arNss={p:0.0 for p in AR_ORDERS_NEW}; self.arNn={p:0 for p in AR_ORDERS_NEW}
        return self

    def update(self, x):
        x=float(x); self.n+=1; n=self.n
        sd_h=self.sd_h; mu_h=self.mu_h; var_h=self.var_h
        d=x-self.mean; dn=d/n; dn2=dn*dn; term=d*dn*(n-1)
        self.mean+=dn
        self.M4+=term*dn2*(n*n-3*n+3)+6*dn2*self.M2-4*dn*self.M3
        self.M3+=term*dn*(n-2)-3*dn*self.M2; self.M2+=term
        var=self.M2/(n-1) if n>1 else 0.0; std=math.sqrt(var) if var>0 else 0.0
        zx=(x-mu_h)/sd_h
        self.cumsum+=zx
        if self.cumsum>self.cmax: self.cmax=self.cumsum
        if self.cumsum<self.cmin: self.cmin=self.cumsum
        cp=self.cusum_p+zx-0.5; cn=self.cusum_n-zx-0.5
        self.cusum_p=cp if cp>0 else 0.0; self.cusum_n=cn if cn>0 else 0.0
        if self.cusum_p>self.cusum_max: self.cusum_max=self.cusum_p
        if self.cusum_n>self.cusum_max: self.cusum_max=self.cusum_n
        azx=zx if zx>=0 else -zx
        if azx>2.0: self.tail2+=1
        if azx>3.0: self.tail3+=1
        if self.prev is not None:
            if abs(x-self.prev)>2.0*sd_h: self.jumps+=1
            if (x-mu_h)*(self.prev-mu_h)<0: self.signruns+=1
            self.sxy+=x*self.prev
        self.sx+=x; self.sxx+=x*x
        lo=bisect_left(self.sorted_ref,x); hi=bisect_right(self.sorted_ref,x)
        self.rank_sum+=0.5*(lo+hi)*self._inv_refn
        self.binc[bisect_right(self.edges,x)]+=1
        if self.ar is not None and len(self.arbuf)==self.p:
            lag=np.array(self.arbuf,dtype=np.float64)[::-1]
            pred=self.ar[0]+self.ar[1:]@lag; r_=x-pred; self.ar_resid_ss+=r_*r_; self.ar_n+=1
        self.arbuf.append(x)
        # --- expanded dependence: higher-lag autocorr products ---
        lb=self.lagbuf
        for L in AC_LAGS_NEW:
            if len(lb)>=L:
                self.acP[L]+=x*lb[-L]; self.acC[L]+=1
        lb.append(x)
        # --- extra AR orders ---
        for p in AR_ORDERS_NEW:
            beta,_vh=self.arN[p]; buf=self.arNbuf[p]
            if beta is not None and len(buf)==p:
                lag=np.array(buf,dtype=np.float64)[::-1]
                pred=beta[0]+beta[1:]@lag; r_=x-pred; self.arNss[p]+=r_*r_; self.arNn[p]+=1
            buf.append(x)
        for a in self.alphas:
            m0=self.ew[a]; self.ew[a]=(1-a)*m0+a*x; self.ewv[a]=(1-a)*(self.ewv[a]+a*(x-m0)**2)
        for w,(buf,acc) in self.roll.items():
            if len(buf)==buf.maxlen: old=buf[0]; acc[0]-=old; acc[1]-=old*old
            buf.append(x); acc[0]+=x; acc[1]+=x*x
        self.specbuf.append(x)
        if len(self.specbuf)==self.SPEC_W:
            p=_norm_psd(np.array(self.specbuf,dtype=np.float64),self._spec_win)
            if p is not None: self._last_spec_ent,self._last_spec_cen=_spec_ec(p,self._spec_freqs,self._spec_logk)
        self.prev=x

        out=self._out; se=sd_h/math.sqrt(n)
        emp=np.cumsum(self.binc)/n; ks=float(np.max(np.abs(emp-self._bin_targets)))
        if n>2 and var>0:
            mx=self.sx/n; cov=self.sxy/(n-1)-mx*mx*n/(n-1); ac1=cov/var
        else: ac1=0.0
        vg=n>=self.VAR_MIN; sqrt_n=math.sqrt(n)
        out[0]=math.log1p(n); out[1]=(self.mean-mu_h)/max(se,1e-8); out[2]=zx
        out[3]=math.log((var+1e-8)/var_h) if vg else 0.0; out[4]=std/sd_h
        out[5]=(self.M3/n)/(var**1.5+1e-8) if vg else 0.0
        out[6]=((self.M4/n)/(var**2+1e-8)-3.0) if vg else 0.0
        out[7]=(self.cmax-self.cmin)/sqrt_n; out[8]=self.cusum_max/sqrt_n
        out[9]=self.tail2/n; out[10]=self.tail3/n; out[11]=self.jumps/n; out[12]=self.signruns/n
        out[13]=ks; out[14]=self.rank_sum/n-0.5; out[15]=ac1; out[16]=ac1-self.ref_ac1
        out[17]=(math.log((self.ar_resid_ss/self.ar_n+1e-8)/(self.ar_var_h+1e-8)) if self.ar_n>=self.VAR_MIN else 0.0)
        out[18]=self.ref_log_n; out[19]=self.ref_skew; out[20]=self.ref_kurt; out[21]=self.ref_ac1
        off=self._ew_off
        for ai,a in enumerate(self.alphas):
            ew_se=sd_h*math.sqrt(a/(2-a))
            out[off+2*ai]=(self.ew[a]-mu_h)/max(ew_se,1e-8)
            out[off+2*ai+1]=math.log((self.ewv[a]+1e-8)/var_h) if vg else 0.0
        off=self._roll_off
        for wi,w in enumerate(self.rw):
            buf,acc=self.roll[w]; L=len(buf); m=acc[0]/L; v=acc[1]/L-m*m
            if v<0: v=0.0
            out[off+2*wi]=(m-mu_h)/sd_h; out[off+2*wi+1]=math.log((v+1e-8)/var_h)
        off=self._spec_off
        out[off]=self._last_spec_ent; out[off+1]=self._last_spec_ent-self.ref_spec_ent
        out[off+2]=self._last_spec_cen-self.ref_spec_cen
        # expanded dependence emit
        off=self._dep_off; j=0; mean_all=self.mean
        for L in AC_LAGS_NEW:
            if self.acC[L]>0 and var>0 and vg:
                acL=(self.acP[L]/self.acC[L]-mean_all*mean_all)/var
            else: acL=0.0
            out[off+j]=acL; out[off+j+1]=acL-self.ref_ac[L]; j+=2
        for p in AR_ORDERS_NEW:
            out[off+j]=(math.log((self.arNss[p]/self.arNn[p]+1e-8)/(self.arN[p][1]+1e-8))
                        if self.arNn[p]>=self.VAR_MIN else 0.0); j+=1
        return out


def _log_steps(L,m):
    if L<=m: return np.arange(L)
    return np.unique(np.round(np.expm1(np.linspace(0,np.log1p(L-1),m))).astype(int))

_ss=StreamState(); COLS=_ss.feature_names; FAMILIES=_ss.FAMILIES
print(len(COLS),"feats. new dependence:",_ss.FAM_DEP_NEW)

43 feats. new dependence: ['ac2_online', 'ac2_shift', 'ac3_online', 'ac3_shift', 'ac5_online', 'ac5_shift', 'ac10_online', 'ac10_shift', 'ar2_resid_logratio', 'ar10_resid_logratio']


## Load + split (6000 train / 2000 holdout)

In [3]:
X=pd.read_parquet(PATH+"X_train.parquet"); Yi=pd.read_parquet(PATH+"y_train_index.parquet")
tau_map=Yi["tau_index"].to_dict()
series=[]
for sid,sub in X.groupby(level="id",sort=False):
    v=sub["value"].to_numpy(); per=sub["period"].to_numpy(); t=int(tau_map[sid])
    series.append((sid,v[per==1],v[per==2],None if t==-1 else t))
train_series=series[:6000]; hold_series=series[6000:8000]
print("train",len(train_series),"hold",len(hold_series))

train 6000 hold 2000


## Extract

In [4]:
SAMPLES_PER=40
def extract(series_list, full):
    st=StreamState(); rows=[]; lab=[]; step=[]; sid_=[]
    for sid,xh,xo,tau in series_list:
        st.reset(xh); L=len(xo)
        want=None if full else set(_log_steps(L,SAMPLES_PER).tolist())
        for k in range(L):
            out=st.update(xo[k])
            if full or k in want:
                rows.append(out.copy()); lab.append(1 if (tau is not None and k>=tau) else 0)
                step.append(k); sid_.append(sid)
    return np.asarray(rows,np.float32),np.asarray(lab,np.int8),np.asarray(step,np.int32),np.asarray(sid_)
t0=time.time(); Xtr,ytr,_,sids=extract(train_series,False); print("train",Xtr.shape,"%.0fs"%(time.time()-t0))
t0=time.time(); Xho,yho,steps,_=extract(hold_series,True); print("hold",Xho.shape,"%.0fs"%(time.time()-t0))

train (197425, 43) 112s


hold (1000961, 43) 41s


## Trainer + TS-AUC

In [5]:
PARAMS=dict(objective="binary",n_estimators=500,learning_rate=0.05,num_leaves=63,
    min_child_samples=300,subsample=0.8,subsample_freq=1,colsample_bytree=0.8,reg_lambda=1.0,n_jobs=-1,verbosity=-1)
cmap=dict(zip(*np.unique(sids,return_counts=True)))
W_TR=np.array([1.0/cmap[s] for s in sids]); W_TR/=W_TR.mean()
def ts_auc(y,sc,st):
    num=den=0.0
    for t in np.unique(st):
        m=st==t; yv=y[m]; npos=int(yv.sum()); nneg=len(yv)-npos
        if npos==0 or nneg==0: continue
        num+=npos*nneg*roc_auc_score(yv,sc[m]); den+=npos*nneg
    return num/den if den else 0.5
def fit_eval(idx,seed=0):
    m=lgb.LGBMClassifier(random_state=seed,**PARAMS).fit(Xtr[:,idx],ytr,sample_weight=W_TR)
    return m, ts_auc(yho,m.predict_proba(Xho[:,idx])[:,1],steps)
n2i={c:i for i,c in enumerate(COLS)}
ALL=list(range(len(COLS)))
m_full,auc_full=fit_eval(ALL)
print("FULL (%d feats) TS-AUC = %.4f"%(len(COLS),auc_full))

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


FULL (43 feats) TS-AUC = 0.5927


## Q1: does expanded dependence beat the shipped set?

Compare full-expanded vs the shipped 33-feat set (drop the 10 new dependence feats).

In [6]:
new_dep=set(_ss.FAM_DEP_NEW)
ship_idx=[i for c,i in n2i.items() if c not in new_dep]
_,auc_ship=fit_eval(ship_idx)
print("shipped 33-feat set     : %.4f"%auc_ship)
print("expanded dependence set : %.4f"%auc_full)
print("delta                   : %+.4f"%(auc_full-auc_ship))

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


shipped 33-feat set     : 0.5862
expanded dependence set : 0.5927
delta                   : +0.0065


## Q2: gain of the new dependence feats

In [7]:
gain=m_full.booster_.feature_importance(importance_type="gain")
rank=pd.DataFrame({"feature":COLS,"gain":gain}); rank["gain_pct"]=100*rank["gain"]/rank["gain"].sum()
rank["is_new_dep"]=rank["feature"].isin(new_dep)
print("new-dep total gain share: %.1f%%"%rank[rank.is_new_dep].gain_pct.sum())
rank.sort_values("gain",ascending=False).reset_index(drop=True).head(25)

new-dep total gain share: 15.5%


,feature,gain,gain_pct,is_new_dep
0,n_online,113506.131360,15.533816,False
1,ref_kurt,69027.425223,9.446708,False
2,ref_skew,61243.574207,8.381454,False
3,ref_log_n,57938.126132,7.929089,False
4,spec_entropy_shift,45853.480051,6.275252,False
5,ref_ac1,44207.874963,6.050043,False
6,spec_entropy,43256.940970,5.919904,False
7,spec_centroid_shift,25905.620114,3.545299,False
8,ar10_resid_logratio,16555.118727,2.265641,True
9,ac5_shift,14944.757291,2.045256,True


## Q3: LOO on the full dependence family (base3 + new10)

In [8]:
drop=set(FAMILIES["dependence"]); keep=[i for c,i in n2i.items() if c not in drop]
_,auc_nodep=fit_eval(keep)
print("full            : %.4f"%auc_full)
print("without ALL dep : %.4f  (delta %+.4f)"%(auc_nodep,auc_nodep-auc_full))

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


full            : 0.5927
without ALL dep : 0.5722  (delta -0.0205)
